In [1]:
!pip install datasets transformers sentence-transformers evaluate rouge-score nltk spacy torch torchvision torchaudio scikit-learn pandas matplotlib seaborn

^C


In [2]:
!python -m spacy download en_core_web_sm

D:\MINI_CONDA\envs\restored_env\python.exe: No module named spacy


In [2]:
import pandas as pd
import numpy as np
import re
import nltk
import spacy
import torch

from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from datasets import load_dataset


KeyboardInterrupt



In [ ]:
nltk.download('stopwords')

In [ ]:
nlp = spacy.load("en_core_web_sm")

stop_words = set(stopwords.words('english'))

In [ ]:
df = pd.read_csv("C:\Users\dipes\Desktop\BBCNews.csv\BBCNews.csv")

df.head()

In [ ]:
df.drop(columns=["Unnamed: 0"], inplace=True)

df.head()

In [ ]:
df = df.rename(columns = {"descr" : 'text',"tags" : 'category'})

In [ ]:
df.head()

In [ ]:
df['category'] = df['category'].apply(
    lambda x: str(x).split(',')[0].strip()
)

In [ ]:
print(df['category'].unique())

In [ ]:
allowed_categories = [
    'sports',
    'business',
    'politics',
    'technology',
    'entertainment'

]

df = df[
    df['category'].isin(allowed_categories)
]

df = df.reset_index(drop=True)

In [ ]:
print(df['category'].value_counts())

In [ ]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"[^a-zA-Z ]", "", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()


def preprocess_text(text):

    text = clean_text(text)

    doc = nlp(text)

    tokens = []

    for token in doc:

        if (
            token.text not in stop_words and not token.is_punct
        ):
            tokens.append(token.lemma_)

    return " ".join(tokens)

    #Stop Word Removal: Deletes common words that don't add much meaning
     #(like "the," "is," or "at").Lemmatization: Reduces words to their base or
    #  dictionary form (e.g., "running" becomes "run," "mice" becomes "mouse").

In [ ]:
df['processed_text'] = df['text'].apply(preprocess_text)

df.head()

In [ ]:
x = df['processed_text']
y = df['category']

x_train, x_temp, y_train, y_temp = train_test_split(
    x, y, test_size=0.3, random_state=42, stratify = y)

x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.5, random_state=42, stratify = y_temp)

print(len(x_train), len(x_val), len(x_test))

In [ ]:
vectorizer = TfidfVectorizer( max_features=10000)

x_train_tfidf = vectorizer.fit_transform(x_train)

x_val_tfidf = vectorizer.transform(x_val)

x_test_tfidf = vectorizer.transform(x_test)


In [ ]:
lr_model = LogisticRegression()
lr_model.fit(x_train_tfidf, y_train)

In [ ]:
val_preds = lr_model.predict(x_val_tfidf)

print(classification_report(y_val, val_preds))

In [ ]:
test_preds = lr_model.predict(x_test_tfidf)

print(classification_report(y_test, test_preds))

In [ ]:
ag_news = load_dataset("SetFit/ag_news")

print(ag_news)

In [ ]:
train_df = pd.DataFrame(ag_news['train'])

test_df = pd.DataFrame(ag_news['test'])

train_df.head()

train_df.drop(columns=["label_text"], inplace=True)

train_df.head()

print(train_df['label'].unique())

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'],
    train_df['label'],
    test_size=0.2,
    random_state=42,
    stratify=train_df['label']
)

In [ ]:
train_texts = train_texts.apply(preprocess_text)

val_texts = val_texts.apply(preprocess_text)

test_df['processed_text'] = test_df['text'].apply(preprocess_text)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
MAX_WORDS = 20000

MAX_LEN = 200

tokenizer = Tokenizer(num_words=MAX_WORDS)

tokenizer.fit_on_texts(train_texts)

In [ ]:
X_train_seq = tokenizer.texts_to_sequences(train_texts)

X_val_seq = tokenizer.texts_to_sequences(val_texts)

X_test_seq = tokenizer.texts_to_sequences(
    test_df['processed_text']
)

In [ ]:
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=MAX_LEN
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN
)

STEP 21: CONVERT TO TENSORS

In [ ]:
X_train_tensor = torch.tensor(X_train_pad)

X_val_tensor = torch.tensor(X_val_pad)

X_test_tensor = torch.tensor(X_test_pad)

y_train_tensor = torch.tensor(train_labels.values)

y_val_tensor = torch.tensor(val_labels.values)

y_test_tensor = torch.tensor(test_df['label'].values)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64
)

In [ ]:
import torch.nn as nn

In [ ]:
class LSTMClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        hidden_dim,
        output_dim
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(
            hidden_dim,
            output_dim
        )

    def forward(self, x):

        embedded = self.embedding(x)

        output, (hidden, cell) = self.lstm(embedded)

        hidden = self.dropout(hidden[-1])

        return self.fc(hidden)

In [ ]:
VOCAB_SIZE = MAX_WORDS

EMBED_DIM = 128

HIDDEN_DIM = 256

OUTPUT_DIM = 4

In [ ]:
model = LSTMClassifier(
    VOCAB_SIZE,
    EMBED_DIM,
    HIDDEN_DIM,
    OUTPUT_DIM
)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.05
)

In [ ]:
EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}")

    print(f"Loss: {total_loss:.4f}")

In [ ]:
model.eval()

all_preds = []
all_labels = []
